# RICAP on Colab

Notebook này dùng trực tiếp repo GitHub để train WideResNet với baseline, RICAP, Mixup hoặc Random Erasing trên CIFAR, đồng thời hỗ trợ các thao tác Git cơ bản ngay trong Colab.

## 1. Clone hoặc cập nhật repo từ GitHub

Nếu repo là private, nhập Personal Access Token ở cell dưới. Nếu repo public, để `github_token = ''`.

In [ ]:
from pathlib import Path
from getpass import getpass
import os
import subprocess

REPO_URL = 'https://github.com/emnguyenddtk21/ricap.git'
REPO_BRANCH = 'master'
WORKSPACE_ROOT = Path('/content/workspace')
REPO_DIR = WORKSPACE_ROOT / 'ricap'

github_token = ''  # hoặc: github_token = getpass('GitHub token: ')

clone_url = REPO_URL
if github_token:
    clone_url = REPO_URL.replace('https://', f'https://{github_token}@')

WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)

def run(cmd):
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)

if not REPO_DIR.exists():
    run(['git', 'clone', '--branch', REPO_BRANCH, clone_url, str(REPO_DIR)])
else:
    run(['git', '-C', str(REPO_DIR), 'remote', 'set-url', 'origin', clone_url])
    run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'])
    run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH])
    run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH])

os.chdir(REPO_DIR)
print('Repo dir:', REPO_DIR)
print('Current branch:')
subprocess.run(['git', 'branch', '--show-current'], check=True)
print('Remote info:')
subprocess.run(['git', 'remote', '-v'], check=True)


## 2. Kiểm tra runtime và cài dependency

In [ ]:
!nvidia-smi || true
!python --version
!pip install -q -r requirements.txt

## 3. Các lệnh Git hữu ích trong Colab

In [ ]:
!git status
!git branch -a
!git log --oneline -5

In [ ]:
# Chạy cell này nếu muốn đồng bộ lại repo sau khi có commit mới trên GitHub
!git fetch origin
!git checkout master
!git pull --ff-only origin master

## 4. Cấu hình train

In [ ]:
dataset = 'cifar10'
use_ricap = True
use_mixup = False
use_random_erase = False

epochs = 200
batch_size = 128
num_workers = 2
pin_memory = True
persistent_workers = True
amp = True

data_dir = '/content/data'
output_dir = '/content/models'
experiment_name = 'colab_ricap_run'

## 5. Train

In [ ]:
import shlex
import subprocess

cmd = [
    'python', 'train.py',
    '--dataset', dataset,
    '--epochs', str(epochs),
    '--batch-size', str(batch_size),
    '--num-workers', str(num_workers),
    '--data-dir', data_dir,
    '--output-dir', output_dir,
    '--name', experiment_name,
]

cmd.append('--pin-memory' if pin_memory else '--no-pin-memory')
cmd.append('--persistent-workers' if persistent_workers else '--no-persistent-workers')
cmd.append('--amp' if amp else '--no-amp')

if use_ricap:
    cmd.append('--ricap')
if use_mixup:
    cmd.append('--mixup')
if use_random_erase:
    cmd.append('--random-erase')

print('Running:')
print(' '.join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)

In [ ]:
import pandas as pd
from pathlib import Path

log_path = Path(output_dir) / experiment_name / 'log.csv'
df = pd.read_csv(log_path)
df.tail()

## 6. Tuỳ chọn: commit và push thay đổi từ Colab

Cell dưới đây dành cho trường hợp bạn chỉnh sửa file ngay trong Colab và muốn đẩy ngược lên GitHub.

In [ ]:
from getpass import getpass
import subprocess

git_user_name = 'Your Name'
git_user_email = 'your_email@example.com'
commit_message = 'Update from Colab'
push_changes = False

subprocess.run(['git', 'config', 'user.name', git_user_name], check=True)
subprocess.run(['git', 'config', 'user.email', git_user_email], check=True)
subprocess.run(['git', 'status', '--short'], check=True)

if push_changes:
    token_for_push = github_token or getpass('GitHub token: ')
    remote_url = REPO_URL.replace('https://', f'https://{token_for_push}@')
    subprocess.run(['git', 'remote', 'set-url', 'origin', remote_url], check=True)
    subprocess.run(['git', 'add', '-A'], check=True)
    subprocess.run(['git', 'commit', '-m', commit_message], check=True)
    subprocess.run(['git', 'push', 'origin', 'master'], check=True)
    print('Push completed.')
else:
    print('Set push_changes = True nếu muốn commit và push từ Colab.')